In [1]:
import time
import platform
import os
import statistics
import numpy as np
import pandas as pd
import duckdb
from pathlib import Path

from evisionary_common import DATA_PATH_MASTER, QUERY_PATTERNS, valid_text_series

KEYB_OUTPUT_DIR = "/Users/sogand/EVisionary_outputs_keyB"

Path(KEYB_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

DATA_PATH_MASTER = os.path.join(KEYB_OUTPUT_DIR, "unified_EVmetadata_keyB.parquet")
DATA_PATH_ENRICHED = os.path.join(KEYB_OUTPUT_DIR, "unified_EVmetadata_enriched.parquet")

OUT_DIR = Path("/Users/sogand/Desktop/Article_Figures/keyB")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_RUNS = 20          # measured runs per engine per scenario
N_WARMUP = 3         # discarded warm-up runs

# ---------------------------------------------------------
# Load data and register DuckDB view
# ---------------------------------------------------------
df = pd.read_parquet(DATA_PATH_MASTER)
con = duckdb.connect(database=":memory:")
con.execute(f"CREATE VIEW ev AS SELECT * FROM read_parquet('{DATA_PATH_MASTER}')")

print("=" * 80)
print("EVisionary local query benchmarking (key_B)")
print("=" * 80)
print(f"Hardware    : {platform.system()} {platform.release()} {platform.machine()}")
print(f"Python      : {platform.python_version()}")
print(f"pandas      : {pd.__version__}")
print(f"duckdb      : {duckdb.__version__}")
print(f"Dataset     : {len(df):,} rows, "
      f"{os.path.getsize(DATA_PATH_MASTER) / 1e6:.2f} MB")
print(f"Runs/engine : {N_RUNS} (after {N_WARMUP} warm-up runs)")


# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def count_unique_pmids(sub):
    pm = valid_text_series(sub["pmid"])
    return int(pm[pm != ""].nunique())


def time_function(func, n_runs, n_warmup):
    """Run a callable repeatedly and return median and IQR of wall time (s)."""
    for _ in range(n_warmup):
        func()
    timings = []
    for _ in range(n_runs):
        start = time.perf_counter()
        func()
        timings.append(time.perf_counter() - start)
    median = statistics.median(timings)
    q75, q25 = np.percentile(timings, 75), np.percentile(timings, 25)
    return median, (q75 - q25)


def time_duckdb(sql, n_runs, n_warmup):
    """Time a DuckDB query (full materialization via .df())."""
    return time_function(lambda: con.execute(sql).df(), n_runs, n_warmup)


# ---------------------------------------------------------
# Scenario definitions (single canonical definitions from common)
# ---------------------------------------------------------
def q1_pandas():
    return df[valid_text_series(df["molecule_type"]).str.contains(
        QUERY_PATTERNS["Protein"], case=False, regex=True, na=False)]

Q1_SQL = (
    "SELECT * FROM ev WHERE "
    "REGEXP_MATCHES(LOWER(TRIM(CAST(molecule_type AS VARCHAR))), '^protein$')"
)


def q2_pandas():
    return df[(valid_text_series(df["species"]) == "Homo sapiens") &
              (valid_text_series(df["molecule_type"]) == "Protein")]

Q2_SQL = (
    "SELECT * FROM ev WHERE "
    "TRIM(CAST(species AS VARCHAR)) = 'Homo sapiens' "
    "AND TRIM(CAST(molecule_type AS VARCHAR)) = 'Protein'"
)


def q3_pandas():
    return df[valid_text_series(df["sample_name"]).str.contains(
        QUERY_PATTERNS["Plasma"], case=False, regex=True, na=False)]

Q3_SQL = (
    r"SELECT * FROM ev WHERE "
    r"REGEXP_MATCHES(LOWER(TRIM(CAST(sample_name AS VARCHAR))), '\bplasma\b')"
)


def q4_pandas():
    return df[valid_text_series(df["disease"]).str.contains(
        QUERY_PATTERNS["Breast Cancer"], case=False, regex=True, na=False)]

Q4_SQL = (
    r"SELECT * FROM ev WHERE "
    r"REGEXP_MATCHES(LOWER(TRIM(CAST(disease AS VARCHAR))), '\bbreast[-\s]+cancer\b')"
)

def q5_pandas():
    # miRNA-class retrieval
    return df[valid_text_series(df["molecule_type"]).str.contains(
        QUERY_PATTERNS["miRNA"], case=False, regex=True, na=False)]

Q5_SQL = (
    "SELECT * FROM ev WHERE "
    "REGEXP_MATCHES(LOWER(TRIM(CAST(molecule_type AS VARCHAR))), '^mirna$')"
)


def q6_pandas():
    # Lipid-class retrieval (newly available after multi-cargo integration).
    return df[valid_text_series(df["molecule_type"]).str.contains(
        QUERY_PATTERNS["Lipid"], case=False, regex=True, na=False)]

Q6_SQL = (
    "SELECT * FROM ev WHERE "
    "REGEXP_MATCHES(LOWER(TRIM(CAST(molecule_type AS VARCHAR))), '^lipid$')"
)
SCENARIOS = [
    ("Q1", "Molecular-class retrieval (Protein)",
     "molecule_type matches '^protein$'", q1_pandas, Q1_SQL),
    ("Q2", "Species-constrained retrieval",
     "species = Homo sapiens AND molecule_type = Protein", q2_pandas, Q2_SQL),
    ("Q3", "Biofluid-proxy free-text retrieval",
     r"sample_name matches '\bplasma\b'", q3_pandas, Q3_SQL),
    ("Q4", "Disease-linked free-text retrieval",
     r"disease matches '\bbreast[-\s]+cancer\b'", q4_pandas, Q4_SQL),
    ("Q5", "Molecular-class retrieval (miRNA)",
     "molecule_type matches '^mirna$'", q5_pandas, Q5_SQL),
    ("Q6", "Molecular-class retrieval (Lipid)",
     "molecule_type matches '^lipid$'", q6_pandas, Q6_SQL),
]

# ---------------------------------------------------------
# Run benchmark
# ---------------------------------------------------------
results = []

for sid, qclass, qdef, pandas_func, duck_sql in SCENARIOS:
    # Correctness cross-check: both engines should return the same row count.
    pandas_n = len(pandas_func())
    duck_n = len(con.execute(duck_sql).df())
    if pandas_n != duck_n:
        print(f"[WARNING] {sid}: pandas={pandas_n} != duckdb={duck_n} "
              f"(definitions diverge; investigate before publishing).")

    pd_med, pd_iqr = time_function(pandas_func, N_RUNS, N_WARMUP)
    dk_med, dk_iqr = time_duckdb(duck_sql, N_RUNS, N_WARMUP)

    pmids = count_unique_pmids(pandas_func())
    speedup = round(pd_med / dk_med, 2) if dk_med > 0 else None

    results.append({
        "Scenario ID": sid,
        "Query class": qclass,
        "Query definition": qdef,
        "Records returned": pandas_n,
        "Unique PMIDs": pmids,
        "Pandas median (s)": round(pd_med, 5),
        "Pandas IQR (s)": round(pd_iqr, 5),
        "DuckDB median (s)": round(dk_med, 5),
        "DuckDB IQR (s)": round(dk_iqr, 5),
        "Speedup (Pandas/DuckDB)": speedup,
    })

    print(f"\n[{sid}] {qclass}")
    print(f"  records={pandas_n:,}  PMIDs={pmids:,}")
    print(f"  pandas  median={pd_med:.5f}s  IQR={pd_iqr:.5f}s")
    print(f"  duckdb  median={dk_med:.5f}s  IQR={dk_iqr:.5f}s")
    print(f"  speedup={speedup}x")

df_bench = pd.DataFrame(results)
out_csv = OUT_DIR / "Table4_Local_Query_Benchmarking.csv"
df_bench.to_csv(out_csv, index=False)

print("\n" + "=" * 80)
print(df_bench.to_string(index=False))
print(f"\nSaved: {out_csv}")

EVisionary local query benchmarking (key_B)
Hardware    : Darwin 25.5.0 arm64
Python      : 3.13.9
pandas      : 2.2.3
duckdb      : 1.4.4
Dataset     : 258,460 rows, 5.52 MB
Runs/engine : 20 (after 3 warm-up runs)

[Q1] Molecular-class retrieval (Protein)
  records=207,623  PMIDs=569
  pandas  median=0.16104s  IQR=0.00674s
  duckdb  median=0.31879s  IQR=0.01307s
  speedup=0.51x

[Q2] Species-constrained retrieval
  records=161,026  PMIDs=413
  pandas  median=0.19662s  IQR=0.00867s
  duckdb  median=0.27746s  IQR=0.02114s
  speedup=0.71x

[Q3] Biofluid-proxy free-text retrieval
  records=547  PMIDs=444
  pandas  median=0.09695s  IQR=0.00366s
  duckdb  median=0.04474s  IQR=0.00277s
  speedup=2.17x

[Q4] Disease-linked free-text retrieval
  records=12  PMIDs=11
  pandas  median=0.10201s  IQR=0.00422s
  duckdb  median=0.05201s  IQR=0.00411s
  speedup=1.96x

[Q5] Molecular-class retrieval (miRNA)
  records=16,131  PMIDs=120
  pandas  median=0.11621s  IQR=0.02684s
  duckdb  median=0.04408s  